In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, TimestampType, DoubleType, IntegerType
from pyspark.sql.functions import col, trim, round

In [0]:
df = spark.table("workspace.bronze.order_items")


In [0]:
RENAME_MAP = {
    "shipping_limit_date": "shipping_deadline"
}
for old_name, new_name in RENAME_MAP.items():
    if old_name in df.columns:
        df = df.withColumnRenamed(old_name, new_name)

In [0]:
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
for col_name in string_cols:
    df = df.withColumn(col_name, trim(col(col_name)))

In [0]:
df = df.filter(
    col("order_id").isNotNull() &
    col("order_item_id").isNotNull() &
    col("product_id").isNotNull()
)


In [0]:
df = (
    df
    # cast to double and round to 2 decimals
    .withColumn("price", round(F.when(col("price").cast(DoubleType()).isNotNull(), col("price").cast(DoubleType())).otherwise(0.0), 2))
    .withColumn("freight_value", round(F.when(col("freight_value").cast(DoubleType()).isNotNull(), col("freight_value").cast(DoubleType())).otherwise(0.0), 2))
    
    # Shipping deadline: cast to Timestamp, null if invalid
    .withColumn("shipping_deadline", F.to_timestamp(col("shipping_deadline")))
    
    # Order item id: cast to integer, default to 0 if invalid
    .withColumn("order_item_id", F.when(col("order_item_id").cast(IntegerType()).isNotNull(), col("order_item_id").cast(IntegerType())).otherwise(0))
)


In [0]:
df = df.dropDuplicates(["order_id", "order_item_id", "product_id"])

In [0]:
df = df.withColumn("silver_processed_time", F.current_timestamp())

In [0]:
df.limit(10).display()  

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.order_items")